In [ ]:
# a) Tabla con un reglón por estación - hora y una columna por contaminante
# b) Covarianza y correlación de CO con NOX ¿Son distintas?
# c) Eigenvalores de la matriz de correlación, con y sin faltantes
# d) Predecir PM2.5 a partir de los demás contaminantes, luego contra solo gases. CalcularR^y VIF
# e) H con una estación y una semana: comprueben H^2 = H y traza = p. Luego el leverage todos sin construir H
# f) Cook: ¿los 10 más influyentes son los de más leverage? Reajusten sin ellos.

In [1]:
# a) Tabla con un reglón por estación - hora y una columna por contaminante

import pandas as pd
import numpy as np

df = pd.read_csv("contaminantes_2026.csv", skiprows=9)

df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
df.loc[df["valor"] < 0, "valor"] = np.nan   # centinelas tipo -99
df["date"] = pd.to_datetime(df["date"], format="mixed")

tabla = (
    df.pivot_table(
        index=["id_station", "date"],
        columns="id_parameter",
        values="valor",
        aggfunc="mean",
    )
    .reset_index()
)
tabla.columns.name = None

print(tabla.head())
print(tabla.describe().T[["count", "mean", "std"]])  # verifica que ya no haya std = 0

  id_station                date  CO  NO  NO2  NOX    O3  PM10  PM2.5  PMCO  \
0        ACO 2026-01-07 00:00:00 NaN NaN  NaN  NaN  24.0   NaN    NaN   NaN   
1        ACO 2026-01-07 01:00:00 NaN NaN  NaN  NaN  22.0   NaN    NaN   NaN   
2        ACO 2026-01-07 02:00:00 NaN NaN  NaN  NaN  14.0   NaN    NaN   NaN   
3        ACO 2026-01-07 03:00:00 NaN NaN  NaN  NaN  15.0   NaN    NaN   NaN   
4        ACO 2026-01-07 04:00:00 NaN NaN  NaN  NaN  11.0   NaN    NaN   NaN   

   SO2  
0  NaN  
1  NaN  
2  NaN  
3  NaN  
4  NaN  
          count                           mean        std
date     133822  2026-04-18 13:39:14.168970752        NaN
CO     111097.0                        0.53659   0.402735
NO      99167.0                      11.885758  25.376202
NO2    110119.0                       23.64097  13.544325
NOX     99168.0                      35.852049  33.902548
O3     124470.0                      34.435342  28.903895
PM10    76201.0                       39.56679  25.348454
PM2.5  

In [2]:
# b) Covarianza y correlación de CO con NOX ¿Son distintas?
sub = tabla[["CO", "NOX"]].dropna()
cov = sub["CO"].cov(sub["NOX"])
corr = sub["CO"].corr(sub["NOX"])
print(f"n = {len(sub)}  cov = {cov:.4f}  corr = {corr:.4f}")
# Son distintas: la covarianza depende de las unidades (ppm vs ppb),
# la correlación es adimensional y siempre está entre -1y 1.

n = 92392  cov = 11.5867  corr = 0.8656


In [3]:
# c) Eigenvalores de la matriz de correlación, con y sin faltantes
cols = ["CO", "NO", "NO2", "NOX", "O3", "PM10", "PM2.5", "PMCO", "SO2"]
datos = tabla[cols]

R_con = datos.corr()
R_sin = datos.dropna().corr()

ev_con = np.linalg.eigvalsh(R_con.values)[::-1]
ev_sin = np.linalg.eigvalsh(R_sin.values)[::-1]

print(pd.DataFrame({"con faltantes (pares)": ev_con,
                    "sin faltantes (completas)": ev_sin}))
print("filas completas:", len(datos.dropna()), "de", len(datos))
# Con pares la matriz puede no ser semidefinida positiva (eigenvalores negativos).
# Con filas completas siempre lo es.

   con faltantes (pares)  sin faltantes (completas)
0               4.303344                   4.455667
1               1.992863                   1.978269
2               0.918607                   0.912216
3               0.657686                   0.644521
4               0.558406                   0.552580
5               0.428957                   0.334203
6               0.143673                   0.122036
7               0.000074                   0.000378
8              -0.003612                   0.000131
filas completas: 17955 de 133822


In [4]:
# d) Predecir PM2.5 a partir de los demás contaminantes, luego contra solo gases. CalcularR^y VIF
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

todos = ["CO", "NO", "NO2", "NOX", "O3", "PM10", "PMCO", "SO2"]
gases = ["CO", "NO", "NO2", "NOX", "O3", "SO2"]

def ajustar(y_col, x_cols):
    d = tabla[[y_col] + x_cols].dropna()
    X, y = d[x_cols].values, d[y_col].values
    modelo = LinearRegression().fit(X, y)
    r2 = r2_score(y, modelo.predict(X))
    # VIF_j = 1 / (1 - R²_j), con R²_j de regresar la variable j contra las demás
    vif = {}
    for j, c in enumerate(x_cols):
        otros = np.delete(X, j, axis=1)
        r2_j = LinearRegression().fit(otros, X[:, j]).score(otros, X[:, j])
        vif[c] = 1 / (1 - r2_j) if r2_j < 1 else np.inf
    return modelo, r2, pd.Series(vif), len(d)

for nombre, xs in [("todos los contaminantes", todos), ("solo gases", gases)]:
    m, r2, vif, n = ajustar("PM2.5", xs)
    print(f"\n== {nombre} (n = {n}) ==  R² = {r2:.4f}")
    print(vif.round(2).to_string())

# Ojo: PMCO = PM10 - PM2.5 por definición, así que con PM10 y PMCO juntos
# el R² será ~1 y sus VIF enormes. NOX ≈ NO + NO2 también infla esos tres VIF.


== todos los contaminantes (n = 17955) ==  R² = 0.9983
CO         5.71
NO      2324.24
NO2      780.73
NOX     4528.15
O3         1.77
PM10       4.54
PMCO       3.40
SO2        1.11

== solo gases (n = 32686) ==  R² = 0.3167
CO        4.23
NO     2343.29
NO2     680.41
NOX    4233.45
O3        1.34
SO2       1.04


In [7]:
# e) H con una estación y una semana: comprueben H^2 = H y traza = p. Luego el leverage todos sin construir H
d = tabla[["id_station", "date", "PM2.5"] + gases].dropna()

# Una estacióny una semana
st = d["id_station"].value_counts().idxmax()
s = d[d["id_station"] == st].sort_values("date")
ini = s["date"].min()
w = s[(s["date"] >= ini) & (s["date"] < ini + pd.Timedelta(days=7))]

Xw = np.column_stack([np.ones(len(w)), w[gases].values])
H = Xw @ np.linalg.pinv(Xw)

p = np.linalg.matrix_rank(Xw)
print(f"estación {st}, n = {len(w)}, columnas de X = {Xw.shape[1]}, rango = {p}")
print("H² = H :", np.allclose(H @ H, H))
print(f"traza(H) = {np.trace(H):.6f}  (debe ser p = {p})")

# Leverage de todos SIN construir
Xall = np.column_stack([np.ones(len(d)), d[gases].values])
Q, _ = np.linalg.qr(Xall)
h = (Q ** 2).sum(axis=1)
print(f"\nsuma de leverages = {h.sum():.4f} (p = {Xall.shape[1]}), max = {h.max():.4f}")

# el mismo método en la semana coincide con diag(H)
Qw, _ = np.linalg.qr(Xw)
print("coincide con diag(H):", np.allclose((Qw ** 2).sum(axis=1), np.diag(H)))

estación UAX, n = 164, columnas de X = 7, rango = 7
H² = H : True
traza(H) = 7.000000  (debe ser p = 7)

suma de leverages = 7.0000 (p = 7), max = 0.0182
coincide con diag(H): True


In [6]:
# f) Cook: ¿los 10 más influyentes son los de más leverage? Reajusten sin ellos.
d = tabla[["PM2.5"] + gases].dropna().reset_index(drop=True)
X, y = d[gases].values, d["PM2.5"].values
n, p = len(d), X.shape[1] + 1

mod = LinearRegression().fit(X, y)
res = y - mod.predict(X)

Xc = np.column_stack([np.ones(n), X])
Q, _ = np.linalg.qr(Xc)
h = (Q ** 2).sum(axis=1)

s2 = res @ res / (n - p)
with np.errstate(divide="ignore", invalid="ignore"):
    cook = (res ** 2 / (p * s2)) * h / (1 - h) ** 2

top_cook = set(np.argsort(cook)[-10:])
top_lev = set(np.argsort(h)[-10:])
print("Top 10 Cook ∩ top 10 leverage:", len(top_cook & top_lev), "de 10")
print(pd.DataFrame({"cook": cook[list(top_cook)], "leverage": h[list(top_cook)]},
                   index=sorted(top_cook)).sort_values("cook", ascending=False))

# Reajuste sin los 10 más influyentes
mask = np.ones(n, dtype=bool)
mask[list(top_cook)] = False
mod2 = LinearRegression().fit(X[mask], y[mask])

print(pd.DataFrame({"con todos": mod.coef_, "sin los 10": mod2.coef_}, index=gases))
print(f"intercepto: {mod.intercept_:.4f} -> {mod2.intercept_:.4f}")
print(f"R²: {mod.score(X, y):.4f} -> {mod2.score(X[mask], y[mask]):.4f}")
# Cook combina leverage Y residuo: un punto puede tener leverage alto
# y no ser influyente si su residuo es pequeño.

Top 10 Cook ∩ top 10 leverage: 0 de 10
           cook  leverage
21045  0.261088  0.002685
21042  0.253419  0.002230
21047  0.194491  0.002592
13919  0.178968  0.001858
21043  0.122915  0.001584
21049  0.096208  0.001715
21044  0.080198  0.003611
21048  0.074228  0.000906
21050  0.069010  0.000846
13918  0.061092  0.002088
     con todos  sin los 10
CO   11.531709    8.690767
NO   -0.074357   -0.028392
NO2   0.200403    0.281031
NOX   0.052246    0.020886
O3    0.165720    0.169183
SO2   0.355599    0.350826
intercepto: 1.5865 -> 1.6145
R²: 0.3167 -> 0.3424
